In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

In [2]:
df_train = pd.read_csv('train.csv')

In [3]:
df_train['target'].value_counts()

target
0    781927
1      4504
Name: count, dtype: int64

In [4]:
df_test = pd.read_csv('test.csv')
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262144 entries, 0 to 262143
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_time  262144 non-null  object 
 1   merch             262144 non-null  object 
 2   cat_id            262144 non-null  object 
 3   amount            262144 non-null  float64
 4   name_1            262144 non-null  object 
 5   name_2            262144 non-null  object 
 6   gender            262144 non-null  object 
 7   street            262144 non-null  object 
 8   one_city          262144 non-null  object 
 9   us_state          262144 non-null  object 
 10  post_code         262144 non-null  int64  
 11  lat               262144 non-null  float64
 12  lon               262144 non-null  float64
 13  population_city   262144 non-null  int64  
 14  jobs              262144 non-null  object 
 15  merchant_lat      262144 non-null  float64
 16  merchant_lon      26

In [5]:
df_train

,transaction_time,merch,cat_id,amount,name_1,name_2,gender,street,one_city,us_state,post_code,lat,lon,population_city,jobs,merchant_lat,merchant_lon,target
0,2019-12-27 15:21,fraud_Cormier LLC,health_fitness,148.04,Daniel,Martinez,M,8510 Acevedo Burgs,Kent,OR,97033,45.0838,-120.6649,60,Museum education officer,45.042827,-120.709327,0
1,2019-04-17 23:09,"fraud_Brown, Homenick and Lesch",health_fitness,39.40,Grace,Williams,F,28812 Charles Mill Apt. 628,Plantersville,AL,36758,32.6176,-86.9475,1412,Drilling engineer,31.872266,-87.828247,0
2,2019-09-23 15:02,fraud_Ruecker-Mayert,kids_pets,52.96,Kyle,Park,M,7507 Larry Passage Suite 859,Mount Perry,OH,43760,39.8788,-82.1880,1831,Barrister's clerk,40.010874,-81.841249,0
3,2019-05-13 16:00,"fraud_Mante, Luettgen and Hackett",health_fitness,7.66,Monique,Martin,F,68276 Matthew Springs,Ratcliff,TX,75858,31.3833,-95.0619,43,"Engineer, production",30.888406,-95.141609,0
4,2019-08-18 07:27,fraud_Luettgen PLC,gas_transport,51.59,Christine,Johnson,F,8011 Chapman Tunnel Apt. 568,Blairsden-Graeagle,CA,96103,39.8127,-120.6405,1725,Chartered legal executive (England and Wales),39.376017,-121.311691,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
786426,2019-04-10 12:35,"fraud_O'Connell, Botsford and Hand",home,76.56,Ryan,Foster,M,03921 Cole Mission Suite 882,Hampton,FL,32044,29.8575,-82.1483,2060,Oncologist,29.235257,-82.407844,0
786427,2019-12-15 09:34,"fraud_Adams, Kovacek and Kuhlman",grocery_net,68.58,Jim,Johnson,M,868 Brady Mill Apt. 837,Gretna,LA,70056,29.8872,-90.0331,55581,Biomedical scientist,29.015274,-90.564712,0
786428,2019-10-12 10:22,"fraud_Lind, Huel and McClure",gas_transport,66.66,Christopher,Horn,M,956 Sanchez Highway,Mallie,KY,41836,37.2692,-82.9161,798,Facilities manager,37.515508,-82.443788,0
786429,2019-10-18 09:01,fraud_Rempel PLC,grocery_net,38.06,Samuel,Sandoval,M,0005 Morrison Land,Mounds,OK,74047,35.8896,-96.0887,7163,Fitness centre manager,35.203864,-96.999902,0


In [6]:
df_train.describe()

,amount,post_code,lat,lon,population_city,merchant_lat,merchant_lon,target
count,786431.000000,786431.000000,786431.000000,786431.000000,7.864310e+05,786431.000000,786431.000000,786431.000000
mean,70.241296,48802.521336,38.527972,-90.224069,8.928853e+04,38.527301,-90.224508,0.005727
std,161.091489,26896.564152,5.078756,13.754760,3.028600e+05,5.113222,13.766977,0.075461
min,1.000000,1257.000000,20.027100,-165.672300,2.300000e+01,19.027804,-166.670132,0.000000
25%,9.650000,26237.000000,34.620500,-96.798000,7.430000e+02,34.727480,-96.901593,0.000000
50%,47.410000,48174.000000,39.346500,-87.476900,2.456000e+03,39.357665,-87.436919,0.000000
75%,83.000000,72042.000000,41.894800,-80.158000,2.047800e+04,41.950609,-80.233429,0.000000
max,27390.120000,99783.000000,66.693300,-67.950300,2.906700e+06,67.441518,-66.955996,1.000000


In [7]:
df_train['transaction_time'] = pd.to_datetime(df_train['transaction_time'])

df_train['year'] = df_train['transaction_time'].dt.year
df_train['month'] = df_train['transaction_time'].dt.month
df_train['day'] = df_train['transaction_time'].dt.day
df_train['hour'] = df_train['transaction_time'].dt.hour
df_train['minute'] = df_train['transaction_time'].dt.minute
df_train['day_of_week'] = df_train['transaction_time'].dt.dayofweek

In [8]:
df_test['transaction_time'] = pd.to_datetime(df_test['transaction_time'])

df_test['year'] = df_test['transaction_time'].dt.year
df_test['month'] = df_test['transaction_time'].dt.month
df_test['day'] = df_test['transaction_time'].dt.day
df_test['hour'] = df_test['transaction_time'].dt.hour
df_test['minute'] = df_test['transaction_time'].dt.minute
df_test['day_of_week'] = df_test['transaction_time'].dt.dayofweek

In [9]:
numeric_features = [
    'amount', 'ts_transaction_time', 'lat', 'lon', 'population_city', 'merchant_lat', 'merchant_lon'
]

cat_columns = [
    'merch', 'cat_id', 'name_1', 'name_2', 'gender', 'street', 'one_city', 'us_state', 'post_code', 'jobs', 'year',	'month',	'day',	'hour',	'minute'	,'day_of_week'
]

In [10]:
import numpy as np
from math import atan2, cos, radians, sin, sqrt


def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float, n_digits: int = 0) -> float:
    """
        Функция для расчёта расстояния от точки А до Б по прямой

        :param lat1: Широта точки А
        :param lon1: Долгота точки А
        :param lat2: Широта точки Б
        :param lon2: Долгота точки Б
        :param n_digits: Округляем полученный ответ до n знака после запятой
        :return: Дистанция по прямой с точностью до n_digits
    """

    lat1, lon1, lat2, lon2 = round(lat1, 6), round(lon1, 6), round(lat2, 6), round(lon2, 6)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)

    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2

    return round(2 * 6372800 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)), n_digits)  # метры.сантиметры


def bearing_degree(lat1: float, lon1: float, lat2: float, lon2: float, n_digits: int = 0) -> float:
    """
        Функция для расчёта угла между прямой [((lat1, lon1), (lat2, lon2)), (нулевой мередиан)]

        :param lat1: Широта точки А
        :param lon1: Долгота точки А
        :param lat2: Широта точки Б
        :param lon2: Долгота точки Б
        :param n_digits: Округляем полученный ответ до n знака после запятой
        :return: Значение угла с точностью до n_digits
    """

    lat1, lon1 = np.radians(round(lat1, 6)), np.radians(round(lon1, 6))
    lat2, lon2 = np.radians(round(lat2, 6)), np.radians(round(lon2, 6))

    dlon = (lon2 - lon1)
    numerator = np.sin(dlon) * np.cos(lat2)
    denominator = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dlon))

    theta = np.arctan2(numerator, denominator)
    theta_deg = (np.degrees(theta) + 360) % 360

    return round(theta_deg, n_digits)

In [11]:
df_train['bearing_degree_1'] = bearing_degree(df_train['lat'], df_train['lon'], df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['bearing_degree_1'] = bearing_degree(df_test['lat'], df_test['lon'], df_test['merchant_lat'], df_test['merchant_lon'], ).values

In [12]:
df_train['bearing_degree_2'] = bearing_degree(df_train['lat'], df_train['lon'], 0, 0, ).values
df_test['bearing_degree_2'] = bearing_degree(df_test['lat'], df_test['lon'], 0, 0, ).values

df_train['bearing_degree_3'] = bearing_degree(0, 0, df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['bearing_degree_3'] = bearing_degree(0, 0, df_test['merchant_lat'], df_test['merchant_lon'], ).values

In [13]:
df_train['hav_dist_1'] = haversine_distance(df_train['lat'], df_train['lon'], df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['hav_dist_1'] = haversine_distance(df_test['lat'], df_test['lon'], df_test['merchant_lat'], df_test['merchant_lon'], ).values
df_train['hav_dist_2'] = haversine_distance(df_train['lat'], df_train['lon'], 0, 0, ).values
df_test['hav_dist_2'] = haversine_distance(df_test['lat'], df_test['lon'], 0, 0, ).values

df_train['hav_dist_3'] = haversine_distance(0, 0, df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['hav_dist_3'] = haversine_distance(0, 0, df_test['merchant_lat'], df_test['merchant_lon'], ).values

In [14]:
model_features = [
  'amount',
  'bearing_degree_1',
  'bearing_degree_2',
  'bearing_degree_3',
  'cat_id',
  'gender',
  'hav_dist_1',
  'hav_dist_2',
  'hav_dist_3',
  'jobs',
  'lat',
  'lon',
  'merch',
  'merchant_lat',
  'merchant_lon',
  'name_1',
  'name_2',
  'one_city',
  'population_city',
  'post_code',
  'street',
  'us_state',
  'year',	
  'month',	
  'day',	
  'hour',	
  'minute',
  'day_of_week'
]

In [15]:
X = df_train.drop("target", axis=1) 
y = df_train["target"]

In [16]:
#cat_features_indices = [X.columns.get_loc(col) for col in cat_features if col in X.columns]
#cat_features_indices

In [17]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X[model_features], y)
print("Распределение классов после RandomOverSampler:")
print(pd.Series(y_train_res).value_counts())

Распределение классов после RandomOverSampler:
target
0    781927
1    781927
Name: count, dtype: int64


In [18]:
X_train_res.columns

Index(['amount', 'bearing_degree_1', 'bearing_degree_2', 'bearing_degree_3',
       'cat_id', 'gender', 'hav_dist_1', 'hav_dist_2', 'hav_dist_3', 'jobs',
       'lat', 'lon', 'merch', 'merchant_lat', 'merchant_lon', 'name_1',
       'name_2', 'one_city', 'population_city', 'post_code', 'street',
       'us_state', 'year', 'month', 'day', 'hour', 'minute', 'day_of_week'],
      dtype='object')

In [19]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim

# --- Data Preparation ---
# Assume you have a DataFrame 'df' with your data and a target variable column 'target'
# For example, loading from a CSV:
# df = pd.read_csv('your_data.csv')


# Label Encoding "Gender" (assuming the gender column is at index 2)
# Label Encoding for "gender" (column index 5)
le_gender = LabelEncoder()
X_train_res['gender'] = le_gender.fit_transform(X_train_res['gender'])

le_city = LabelEncoder()
X_train_res['one_city'] = le_city.fit_transform(X_train_res['one_city'])

In [20]:
X_train_res.columns

Index(['amount', 'bearing_degree_1', 'bearing_degree_2', 'bearing_degree_3',
       'cat_id', 'gender', 'hav_dist_1', 'hav_dist_2', 'hav_dist_3', 'jobs',
       'lat', 'lon', 'merch', 'merchant_lat', 'merchant_lon', 'name_1',
       'name_2', 'one_city', 'population_city', 'post_code', 'street',
       'us_state', 'year', 'month', 'day', 'hour', 'minute', 'day_of_week'],
      dtype='object')

In [21]:
X_train_res['population_city']

0             60
1           1412
2           1831
3             43
4           1725
           ...  
1563849    22305
1563850     3807
1563851      741
1563852      137
1563853     2379
Name: population_city, Length: 1563854, dtype: int64

In [22]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# Let's assume X_train_res is your feature matrix (a 2D numpy array or convertible to one)
# and y_train_res is your target array.

# Define the categorical column indices:
categorical_columns = [4, 9, 19, 20, 21, 12,15,16]

# Create the ColumnTransformer.
# We set sparse=False so the encoder returns a dense array.
ctCategory = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(), categorical_columns)],
    remainder='passthrough'
)

# Apply the transformer to your feature matrix.
X_res_encoded = ctCategory.fit_transform(X_train_res)

# Now split the encoded features and target into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(X_res_encoded, y_train_res, test_size=0.2, random_state=0)


: 

In [ ]:


X_train = X_train.toarray()  # or use .todense() if you prefer
X_test = X_test.toarray()

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# --- Building the PyTorch Neural Network ---
class Net(nn.Module):
    def __init__(self, input_dim):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_dim, 6)
        self.fc2 = nn.Linear(6, 6)
        self.fc3 = nn.Linear(6, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.relu(self.fc2(out))
        out = self.sigmoid(self.fc3(out))
        return out



In [ ]:
input_dim = X_train_tensor.shape[1]
model = Net(input_dim)

In [ ]:


# Define loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Training the Model ---
num_epochs = 33
batch_size = 32
num_samples = X_train_tensor.shape[0]

for epoch in range(num_epochs):
    permutation = torch.randperm(num_samples)
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train_tensor[indices], y_train_tensor[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")
# --- Evaluating on the Test Set ---
model.eval()  # Set model to evaluation mode
with torch.no_grad():
    y_pred = model(X_test_tensor)
    # Convert probabilities to binary predictions (threshold=0.5)
    y_pred_label = (y_pred > 0.5).float()
    # Concatenate predictions and true labels for inspection
    results = torch.cat((y_pred_label, y_test_tensor), dim=1)
    print(results)



KeyboardInterrupt: 

In [30]:


# --- Predicting a New Sample ---
# Define a new customer sample (adjust the order and values according to your features)
customer_test = [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0,
                 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                 10.0, 0, 157, 333497, 1992]

# Make sure the new sample has the same shape as your original X data (after one-hot encoding and preprocessing)
# If needed, you might have to apply the same transformations to this new sample.
customer_test = np.array(customer_test).reshape(1, -1)
customer_test_scaled = sc.transform(customer_test)
customer_test_tensor = torch.tensor(customer_test_scaled, dtype=torch.float32)

with torch.no_grad():
    prob_leave = model(customer_test_tensor)
    print(prob_leave > 0.5)


InvalidIndexError: (slice(None, None, None), 5)

In [21]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'iterations': [200, 500],
    'learning_rate': [ 0.1, 0.03],
    'depth': [1, 2, 3, 5, 7]
}

clf = CatBoostClassifier(
    verbose=False,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=1 
)


best_model = grid_search.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_columns, early_stopping_rounds=50)


print("Лучшие гиперпараметры:", grid_search.best_params_)
print("Лучший результат кросс-валидации:", grid_search.best_score_)


KeyboardInterrupt: 

In [ ]:
val_score = best_model.score(X_val, y_val)
print("Результат на валидационной выборке:", val_score)

Результат на валидационной выборке: 0.999994884439488


лучшие параметры iterations=200, learning_rate=0.1, depth=3,

In [22]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(
    iterations=500,        # выбранный параметр
    learning_rate=0.01,       # выбранный параметр
    depth=3,                 # выбранный параметр
    random_state=42,
    verbose=False
)


clf.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_columns, early_stopping_rounds=50)


val_score = clf.score(X_val, y_val)
print("Результат на валидационной выборке:", val_score)


KeyboardInterrupt: 

In [30]:
df_test=df_test[model_features]

In [31]:
y_pred = clf.predict(df_test)

In [32]:
submission = pd.DataFrame({'prediction': y_pred})

submission = submission.reset_index()

submission.rename(columns={'index': 'index', 'prediction': 'prediction'}, inplace=True)

submission.to_csv("submission_4.csv", index=False)

In [33]:
submission['prediction'].value_counts()

prediction
0    261171
1       973
Name: count, dtype: int64

НЕ ЗАБЫТЬ ПРОВЕРИТЬ!!!!!!!!!!